In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [3]:
# ============================================================
# Import Libraries
# ============================================================

from pathlib import Path
import string

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import (
    ENGLISH_STOP_WORDS,
    TfidfVectorizer
)

from sklearn.metrics.pairwise import cosine_similarity

In [4]:
# ============================================================
# Dataset Paths
# ============================================================

PROJECT_DIR = Path.cwd()

KAGGLE_DATA_DIR = Path(
    "/kaggle/input/competitions/smart-mcq-solver-challenge"
)

if KAGGLE_DATA_DIR.exists():
    DATA_DIR = KAGGLE_DATA_DIR
    OUTPUT_DIR = Path("/kaggle/working")
else:
    DATA_DIR = PROJECT_DIR / "data"
    OUTPUT_DIR = PROJECT_DIR / "outputs"

OPTION_LABELS = np.array(list("ABCDE"))

In [5]:
# ============================================================
# Load Dataset
# ============================================================

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

print(train.shape)
print(test.shape)

train.head()

(2000, 8)
(500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [6]:
# ============================================================
# Helper Functions
# ============================================================

def clean_prompt(text):
    return text.lower().translate(
        str.maketrans("", "", string.punctuation)
    )


def combined_text(df):
    columns = ["prompt", *OPTION_LABELS]
    return df[columns].fillna("").agg(" ".join, axis=1)


def option_similarities(vectorizer, df):

    similarities = []

    for _, row in df.iterrows():

        prompt_vec = vectorizer.transform([row["prompt"]])

        scores = [
            cosine_similarity(
                prompt_vec,
                vectorizer.transform([row[option]])
            )[0, 0]

            for option in OPTION_LABELS
        ]

        similarities.append(scores)

    return np.asarray(similarities)


def rank_options(similarities):

    return [
        [
            label
            for label, _
            in sorted(
                zip(OPTION_LABELS, row),
                key=lambda x: x[1],
                reverse=True
            )
        ]

        for row in similarities
    ]


def map_at_3(y_true, preds):

    scores = []

    for truth, pred in zip(y_true, preds):

        if truth in pred:
            scores.append(1 / (pred.index(truth) + 1))
        else:
            scores.append(0)

    return np.mean(scores)

In [7]:
# ============================================================
# Exploratory Data Analysis
# ============================================================

answer_counts = train["answer"].value_counts().sort_index()

display(answer_counts)

print("Most + Least Frequent:",
      answer_counts.max() + answer_counts.min())

answer
A    369
B    490
C    459
D    358
E    324
Name: count, dtype: int64

Most + Least Frequent: 814


In [8]:
# ============================================================
# Text Cleaning
# ============================================================

cleaned_prompts = train["prompt"].map(clean_prompt)

vocab = set(
    " ".join(cleaned_prompts).split()
)

print("Vocabulary Size:", len(vocab))

Vocabulary Size: 859


In [9]:
# ============================================================
# Stopword Removal Example
# ============================================================

row_prompt = cleaned_prompts.loc[
    train["id"] == 1
].iloc[0]

filtered_words = [

    word

    for word in row_prompt.split()

    if word not in ENGLISH_STOP_WORDS

]

print(filtered_words)
print(len(filtered_words))

['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']
13


In [10]:
# ============================================================
# TF-IDF Baseline
# ============================================================

vectorizer = TfidfVectorizer(
    stop_words="english"
)

vectorizer.fit(
    combined_text(train)
)

print(
    "Vocabulary Size:",
    len(vectorizer.get_feature_names_out())
)

Vocabulary Size: 2762


In [11]:
# ============================================================
# Cosine Similarity
# ============================================================

train_similarity = option_similarities(
    vectorizer,
    train
)

train_rankings = rank_options(
    train_similarity
)

top1 = np.array([
    row[0]
    for row in train_rankings
])

print(
    "Row 1 Prompt vs Option A:",
    train_similarity[0,0]
)

print(
    "Top-1 Accuracy:",
    (top1 == train.answer).mean()
)

print(
    "MAP@3:",
    map_at_3(train.answer, train_rankings)
)

Row 1 Prompt vs Option A: 0.27202429519891635
Top-1 Accuracy: 0.1355
MAP@3: 0.4020916666666667


In [12]:
# ============================================================
# Majority Baseline
# ============================================================

majority = train.answer.value_counts().index[:3].tolist()

majority_preds = [
    majority
] * len(train)

print(majority)

print(
    map_at_3(
        train.answer,
        majority_preds
    )
)

['B', 'C', 'A']
0.42125


In [13]:
# ============================================================
# Generate Submission
# ============================================================

test_similarity = option_similarities(
    vectorizer,
    test
)

test_rankings = rank_options(
    test_similarity
)

submission = pd.DataFrame({

    "ID": test.id,

    "Prediction": [
        " ".join(row[:3])
        for row in test_rankings
    ]

})

OUTPUT_DIR.mkdir(exist_ok=True)

submission.to_csv(
    OUTPUT_DIR / "submission.csv",
    index=False
)

submission.head()

,ID,Prediction
0,1,A B C
1,2,A B C
2,3,A D C
3,4,A E C
4,5,A C B


# Milestone 2

In [ ]:
"""
Milestone 2 solution script.

pip install -q transformers datasets sentence-transformers torch
"""

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, pipeline
from sentence_transformers import SentenceTransformer, util


In [ ]:

# ---------------------------------------------------------------------
# Q1: combined_text length at index 51
# ---------------------------------------------------------------------
ds = load_dataset("csv", data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")["train"]
ds = ds.map(lambda x: {"combined_text": x["prompt"] + " " + x["A"]})
print("Q1 combined_text length @51:", len(ds[51]["combined_text"]))


In [ ]:
# ---------------------------------------------------------------------
# Q2 & Q3: tokenizer vocab size and [SEP] id
# ---------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
print("Q2 vocab_size:", tokenizer.vocab_size)          # 30522
print("Q3 [SEP] id:", tokenizer.sep_token_id)           # 102


In [ ]:
# ---------------------------------------------------------------------
# Q4: tokenize entire prompt column
# ---------------------------------------------------------------------
prompt_list = [str(p) for p in ds["prompt"]]  # force plain python str
encoded = tokenizer(
    prompt_list, padding="max_length", truncation=True,
    max_length=128, return_tensors="pt"
)
print("Q4 input_ids shape:", encoded["input_ids"].shape)  # [2000, 128]


# ---------------------------------------------------------------------
# Q5: attention head dimensionality
# ---------------------------------------------------------------------
print("Q5 head dim:", 768 // 12)  # 64


In [ ]:



# ---------------------------------------------------------------------
# Q6 & Q7: last_hidden_state shape + CLS vector sum
# ---------------------------------------------------------------------
model = AutoModel.from_pretrained("bert-base-uncased")
model.eval()

row0_prompt = ds[0]["prompt"]
inputs = tokenizer(row0_prompt, return_tensors="pt")  # default settings
with torch.no_grad():
    out = model(**inputs)

print("Q6 last_hidden_state shape:", out.last_hidden_state.shape)

cls_vec = out.last_hidden_state[0, 0, :]  # [CLS] is token index 0
cls_sum_first5 = cls_vec[:5].sum().item()
print("Q7 sum of first 5 CLS values:", round(cls_sum_first5, 4))


In [ ]:
# ---------------------------------------------------------------------
# Q8: attention weight from [CLS] to "fusion"
# ---------------------------------------------------------------------
attn_model = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)
attn_model.eval()

text = "Light-ion fusion is a technique."
attn_inputs = tokenizer(text, return_tensors="pt")
with torch.no_grad():
    attn_out = attn_model(**attn_inputs)
# find token index for "fusion"
tokens = tokenizer.convert_ids_to_tokens(attn_inputs["input_ids"][0])
print("tokens:", list(enumerate(tokens)))
fusion_idx = tokens.index("fusion")

last_layer_attn = attn_out.attentions[-1]        # (batch, heads, seq, seq)
cls_to_fusion = last_layer_attn[0, 0, 0, fusion_idx].item()
print("Q8 [CLS]->fusion attention weight:", round(cls_to_fusion, 4))


In [ ]:
# ---------------------------------------------------------------------
# Q9: MiniLM cosine similarity, prompt vs Option B, row 0
# ---------------------------------------------------------------------
minilm = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
emb_prompt = minilm.encode(ds[0]["prompt"])
emb_b = minilm.encode(ds[0]["B"])
sim = util.cos_sim(emb_prompt, emb_b).item()
print("Q9 cosine similarity prompt vs B (row 0):", round(sim, 4))


In [ ]:

# ---------------------------------------------------------------------
# Q10: MAP@3 — TF-IDF pipeline vs MiniLM pipeline
# ---------------------------------------------------------------------
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

options_cols = ["A", "B", "C", "D", "E"]

def map_at_3(top3_lists, answers):
    scores = []
    for top3, ans in zip(top3_lists, answers):
        if ans not in top3:
            scores.append(0.0)
        else:
            rank = top3.index(ans) + 1
            scores.append(1.0 / rank)
    return sum(scores) / len(scores)



In [ ]:
# --- Pipeline 1: TF-IDF ---
tfidf_top3 = []
for row in ds:
    corpus = [row["prompt"]] + [row[c] for c in options_cols]
    vec = TfidfVectorizer().fit_transform(corpus)
    sims = cosine_similarity(vec[0:1], vec[1:]).flatten()
    order = np.argsort(-sims)
    top3 = [options_cols[i] for i in order[:3]]
    tfidf_top3.append(top3)

# --- Pipeline 2: MiniLM ---
minilm_top3 = []
for row in ds:
    prompt_emb = minilm.encode(row["prompt"])
    opt_embs = minilm.encode([row[c] for c in options_cols])
    sims = util.cos_sim(prompt_emb, opt_embs).flatten().numpy()
    order = np.argsort(-sims)
    top3 = [options_cols[i] for i in order[:3]]
    minilm_top3.append(top3)

answers = ds["answer"]
map3_minilm = map_at_3(minilm_top3, answers)
print("Q10a MAP@3 (MiniLM):", round(map3_minilm, 4))

count_gained = 0
for t3_tfidf, t3_minilm, ans in zip(tfidf_top3, minilm_top3, answers):
    if ans not in t3_tfidf and ans in t3_minilm:
        count_gained += 1
print("Q10b count improved by MiniLM over TF-IDF:", count_gained)



In [14]:
# ---------------------------------------------------------------------
# Q11 & Q12: zero-shot classification
# ---------------------------------------------------------------------
zsc = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

row1 = ds[1]
candidates = [row1["A"], row1["B"], row1["C"]]

result_softmax = zsc(row1["prompt"], candidate_labels=candidates)
top_score = result_softmax["scores"][0]
print("Q11 top-ranked score (softmax):", round(top_score, 4))

result_sigmoid = zsc(row1["prompt"], candidate_labels=candidates, multi_label=True)
sum_softmax = sum(result_softmax["scores"])
sum_sigmoid = sum(result_sigmoid["scores"])
print("Q12 |sum(softmax) - sum(sigmoid)|:", round(abs(sum_softmax - sum_sigmoid), 4))


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Q1 combined_text length @51: 614


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Q13 model output: B


In [ ]:

# ---------------------------------------------------------------------
# Q13: Flan-T5-small generative QA
# ---------------------------------------------------------------------
from transformers import AutoModelForSeq2SeqLM

t5_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
t5_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")
t5_model.eval()

row0 = ds[0]
prompt_str = (
    f"Question: {row0['prompt']}. Is the correct answer "
    f"A: {row0['A']} or B: {row0['B']}? Answer with just the letter A or B."
)
t5_inputs = t5_tokenizer(prompt_str, return_tensors="pt")
with torch.no_grad():
    t5_out_ids = t5_model.generate(**t5_inputs, max_new_tokens=5)
t5_output_text = t5_tokenizer.decode(t5_out_ids[0], skip_special_tokens=True)
print("Q13 model output:", t5_output_text)

# milestone-3

In [2]:
# Install
!pip install -q faiss-cpu sentence-transformers transformers

import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import pipeline, AutoTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from transformers import AutoTokenizer

In [ ]:
# -----------------------------
# Load Dataset
# -----------------------------
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")


In [ ]:

# -----------------------------
# Create Knowledge Base
# -----------------------------
print("Creating knowledge base...")

kb = []

for _, row in train.iterrows():
    correct_letter = row["answer"]      # A/B/C/D/E
    kb.append(str(row[correct_letter])) # Store only the correct answer text

print(f"Knowledge Base Size: {len(kb)}")


In [ ]:


# -----------------------------
# Load Embedding Model
# -----------------------------
print("Loading embedding model...")

model = SentenceTransformer("all-MiniLM-L6-v2")


In [ ]:

# -----------------------------
# Create Embeddings
# -----------------------------
print("Creating embeddings...")

kb_embeddings = model.encode(
    kb,
    show_progress_bar=True,
    convert_to_numpy=True
)


In [ ]:

# -----------------------------
# Build FAISS Index
# -----------------------------
dimension = kb_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(kb_embeddings)

print(f"Indexed {index.ntotal} documents.")


In [ ]:

# -----------------------------
# Load Zero-shot Classifier
# -----------------------------
print("Loading Zero-Shot classifier...")

zs = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)


In [ ]:

# -----------------------------
# Load Cross Encoder
# -----------------------------
print("Loading Cross Encoder...")

cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)


In [16]:

# -----------------------------
# Load BERT Tokenizer
# -----------------------------
print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

# -----------------------------
# Example row 
# -----------------------------
row_150 = train.iloc[150]

prompt_150 = str(row_150["prompt"])

labels_150 = [
    str(row_150["A"]),
    str(row_150["B"]),
    str(row_150["C"]),
    str(row_150["D"]),
    str(row_150["E"])
]

ans_150 = str(row_150[row_150["answer"]])

print("\nSetup Complete!")
print("Knowledge Base Size :", len(kb))
print("Embedding Dimension :", dimension)

Creating knowledge base...
Knowledge Base Size: 2000
Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Creating embeddings...


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Loading tokenizer...

Setup Complete!
Knowledge Base Size : 2000
Embedding Dimension : 384


In [ ]:

# -----------------------------
# Q1
# -----------------------------
result = zs(prompt_150, candidate_labels=labels_150)

score_dict = dict(zip(result["labels"], result["scores"]))
q1 = score_dict[ans_150]

print("Q1 =", round(q1, 3))


In [ ]:
# -----------------------------
# Q2
# -----------------------------
query_embedding = model.encode([prompt_150])

D, I = index.search(query_embedding, 10)

retrieved_indices = I[0]

true_rank = None

for rank, idx in enumerate(retrieved_indices, start=1):
    if idx == 150:
        true_rank = rank
        break

print("Q2 =", true_rank)


In [ ]:
# -----------------------------
# Q3
# -----------------------------
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

docs_10 = [kb[i] for i in retrieved_indices]

pairs = [[prompt_150, doc] for doc in docs_10]

scores = cross_encoder.predict(pairs)

order = np.argsort(scores)[::-1]

reranked_indices = [retrieved_indices[i] for i in order]

true_rank_ce = None

for rank, idx in enumerate(reranked_indices, start=1):
    if idx == 150:
        true_rank_ce = rank
        break

print("Q3 =", true_rank_ce)


In [ ]:
# -----------------------------
# Q4
# -----------------------------
row42 = train.iloc[42]
prompt42 = str(row42["prompt"])

emb42 = model.encode([prompt42])

_, idx42 = index.search(emb42, 5)

docs5 = [kb[i] for i in idx42[0]]

context = " ".join(docs5)

rag = f"Context: {context} Question: {prompt42}"

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

tokens = tokenizer(rag, truncation=False)

print("Q4 =", len(tokens["input_ids"]))


In [ ]:
# -----------------------------
# Q5
# -----------------------------
true_doc = kb[150]

rag_true = f"Context: {true_doc} Question: {prompt_150}"

result = zs(rag_true, candidate_labels=labels_150)

score_dict = dict(zip(result["labels"], result["scores"]))

q5 = score_dict[ans_150]

print("Q5 =", round(q5, 3))


In [ ]:
# -----------------------------
# Q6
# -----------------------------
bad_doc = kb[999]

rag_bad = f"Context: {bad_doc} Question: {prompt_150}"

result = zs(rag_bad, candidate_labels=labels_150)

score_dict = dict(zip(result["labels"], result["scores"]))

q6 = score_dict[ans_150]

print("Q6 =", round(q6, 3))


In [ ]:
# -----------------------------
# Q7
# -----------------------------
hits = 0

for i in range(100):

    row = train.iloc[i]

    prompt = str(row["prompt"])

    correct = str(row[row["answer"]])

    emb = model.encode([prompt])

    _, idx = index.search(emb, 5)

    docs = [kb[j] for j in idx[0]]

    if any(correct in d for d in docs):
        hits += 1

hit_rate = hits / 100 * 100

print("Q7 =", round(hit_rate, 1))


In [17]:

# -----------------------------
# Q8
# -----------------------------
def apk(actual, predicted, k=3):

    score = 0.0
    hits = 0.0

    predicted = predicted[:k]

    for i, p in enumerate(predicted):

        if p == actual:
            hits += 1
            score += hits / (i + 1)

    return score


scores = []

for i in range(20):

    row = train.iloc[i]

    prompt = str(row["prompt"])

    labels = [
        str(row["A"]),
        str(row["B"]),
        str(row["C"]),
        str(row["D"]),
        str(row["E"])
    ]

    letters = ["A","B","C","D","E"]

    emb = model.encode([prompt])

    _, idx = index.search(emb, 5)

    retrieved = idx[0]

    docs = [kb[j] for j in retrieved]

    pairs = [[prompt, d] for d in docs]

    ce = cross_encoder.predict(pairs)

    best_doc = docs[np.argmax(ce)]

    rag = f"Context: {best_doc} Question: {prompt}"

    result = zs(rag, candidate_labels=labels)

    label_scores = dict(zip(result["labels"], result["scores"]))

    option_scores = {}

    for letter in letters:
        option_scores[letter] = label_scores[str(row[letter])]

    ranked = sorted(option_scores.items(),
                    key=lambda x: x[1],
                    reverse=True)

    prediction = [x[0] for x in ranked]

    actual = row["answer"]

    scores.append(apk(actual, prediction, 3))

print("Q8 =", round(np.mean(scores), 3))

Q1 = 0.384
Q2 = 10


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Q3 = 1
Q4 = 216
Q5 = 0.989
Q6 = 0.529
Q7 = 73.0


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Q8 = 0.975
